In [7]:
import sys
from pathlib import Path
project_root = Path().resolve().parent  
sys.path.insert(0, str(project_root))

from pathlib import Path
import typer
import pandas as pd
import numpy as np

from creditcard_psp.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

def synthesize(
    source_path: Path = typer.Option(RAW_DATA_DIR / "PSP_Jan_Feb_2019.xlsx", help="Quelle für Verteilungen"),
    n: int = typer.Option(200, min=1, help="Anzahl synthetischer Zeilen"),
    out_path: Path = typer.Option(PROCESSED_DATA_DIR / "synthetic.parquet", help="Ziel-Datei"),
):
    """
    Erzeugt einfache synthetische Daten via Bootstrap + leichtem Jitter (nur für Tests/Demos).
    """
    df = read_table(source_path)
    # Delete target
    drop_cols = {"transaction_success", "success", "fee_successful", "fee_not_successful"}
    keep = [c for c in df.columns if c not in drop_cols]
    df = df[keep]

    samp = df.sample(n=n, replace=True, random_state=42).reset_index(drop=True)
    # Slight jitter for numeric columns
    for c in samp.select_dtypes(include=[np.number]).columns:
        sd = samp[c].std(ddof=0)
        jitter = 0.01 * (sd if sd > 0 else 1.0)
        samp[c] = samp[c] + np.random.normal(0.0, jitter, size=len(samp))

    samp.to_parquet(out_path, index=False)
    typer.echo(f"Synthetische Daten gespeichert: {out_path} (n={n})")

In [ ]:
synthesize()